#### Imports and setup

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import RAW_DATA_FILE, TARGET_COLUMN
from src.data_loader import load_energy_dataset, parse_datetime_column
from src.feature_engineering_utils import (
    create_time_features,
    create_lag_features,
    create_rolling_features,
    create_interaction_features,
    drop_feature_engineering_nulls,
    save_dataframe,
)

pd.set_option("display.max_columns", None)

## Feature Engineering

This notebook creates additional predictive features for the appliance energy forecasting task.  
The engineered features include time-based features, lagged features, rolling statistics, and interaction features.

#### 1) Load Dataset

In [6]:
energy_df = load_energy_dataset("E:\Vectorium Labs\energy-forecast\dataset\energy_data_set.csv")
energy_df = parse_datetime_column(energy_df, datetime_column="date", set_as_index=True)

energy_df.head()

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,RH_5,T6,RH_6,T7,RH_7,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,45.566667,17.166667,55.20,7.026667,84.256667,17.200000,41.626667,18.2,48.900000,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,45.992500,17.166667,55.20,6.833333,84.063333,17.200000,41.560000,18.2,48.863333,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,45.890000,17.166667,55.09,6.560000,83.156667,17.200000,41.433333,18.2,48.730000,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,45.723333,17.166667,55.09,6.433333,83.423333,17.133333,41.290000,18.1,48.590000,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,45.530000,17.200000,55.09,6.366667,84.893333,17.200000,41.230000,18.1,48.590000,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [7]:
# dataset shape 
print("Shape before feature engineering:", energy_df.shape)

Shape before feature engineering: (19735, 28)


#### 2) Creating time-based features

In [8]:
energy_df = create_time_features(energy_df)

energy_df[
    ["Appliances", "hour", "day_of_week", "month", "day_of_month", "is_weekend"]
].head()

,Appliances,hour,day_of_week,month,day_of_month,is_weekend
date,,,,,,
2016-01-11 17:00:00,60,17,0,1,11,0
2016-01-11 17:10:00,60,17,0,1,11,0
2016-01-11 17:20:00,50,17,0,1,11,0
2016-01-11 17:30:00,50,17,0,1,11,0
2016-01-11 17:40:00,60,17,0,1,11,0


- Time-related variables were created to capture temporal usage patterns in appliance energy consumption.  
- These include hour of the day, day of the week, month, day of the month, and a weekend indicator.

#### 3) Creating lag features

In [9]:
lag_steps = [1, 3, 6] # 10mins, 30mins, 1hr

energy_df = create_lag_features(
    energy_df=energy_df,
    target_column=TARGET_COLUMN,
    lag_steps=lag_steps,
)

energy_df[
    ["Appliances", "Appliances_lag_1", "Appliances_lag_3", "Appliances_lag_6"]
].head(10)

,Appliances,Appliances_lag_1,Appliances_lag_3,Appliances_lag_6
date,,,,
2016-01-11 17:00:00,60,NaN,NaN,NaN
2016-01-11 17:10:00,60,60.0,NaN,NaN
2016-01-11 17:20:00,50,60.0,NaN,NaN
2016-01-11 17:30:00,50,50.0,60.0,NaN
2016-01-11 17:40:00,60,50.0,60.0,NaN
2016-01-11 17:50:00,50,60.0,50.0,NaN
2016-01-11 18:00:00,60,50.0,50.0,60.0
2016-01-11 18:10:00,60,60.0,60.0,60.0
2016-01-11 18:20:00,60,60.0,50.0,50.0



- Lagged target features were created to capture temporal dependency in appliance energy consumption.  
- These features allow the model to use recent historical energy usage when predicting future consumption.

#### 4) Creating rolling features

In [10]:
window_sizes = [6, 12] # 1h rolling mean, 2h rolling mean

energy_df = create_rolling_features(
    energy_df=energy_df,
    target_column=TARGET_COLUMN,
    window_sizes=window_sizes,
)

energy_df[
    [
        "Appliances",
        "Appliances_rolling_mean_6",
        "Appliances_rolling_mean_12",
    ]
].head(15)

,Appliances,Appliances_rolling_mean_6,Appliances_rolling_mean_12
date,,,
2016-01-11 17:00:00,60,NaN,NaN
2016-01-11 17:10:00,60,NaN,NaN
2016-01-11 17:20:00,50,NaN,NaN
2016-01-11 17:30:00,50,NaN,NaN
2016-01-11 17:40:00,60,NaN,NaN
2016-01-11 17:50:00,50,55.000000,NaN
2016-01-11 18:00:00,60,55.000000,NaN
2016-01-11 18:10:00,60,55.000000,NaN
2016-01-11 18:20:00,60,56.666667,NaN


- Rolling mean features were generated to smooth short-term fluctuations and capture local energy consumption trends.  
- These features help represent recent average behavior rather than relying only on single past points.

#### 5) Creating interaction features

In [11]:
energy_df = create_interaction_features(energy_df)

interaction_columns = [
    column_name
    for column_name in energy_df.columns
    if "interaction" in column_name
]

energy_df[interaction_columns].head()

,T_out_RH_out_interaction,T1_RH_1_interaction,T2_RH_2_interaction
date,,,
2016-01-11 17:00:00,607.200000,946.6977,859.968
2016-01-11 17:10:00,596.466667,928.7304,858.672
2016-01-11 17:20:00,585.733333,920.9070,856.832
2016-01-11 17:30:00,575.000000,916.2660,856.128
2016-01-11 17:40:00,564.266667,921.5700,854.976


- Interaction terms were created to capture combined effects between temperature and humidity variables.  
- These combined environmental conditions may influence appliance energy use more effectively than the original variables alone.

#### 6) Checks

In [12]:
# checking for missing values after feature eng process
energy_df.isnull().sum()[energy_df.isnull().sum() > 0]

Appliances_lag_1               1
Appliances_lag_3               3
Appliances_lag_6               6
Appliances_rolling_mean_6      5
Appliances_rolling_mean_12    11
dtype: int64

In [13]:
# removing rows with null values (caused by the lag or rolling)
energy_df = drop_feature_engineering_nulls(energy_df)

print("Shape after dropping nulls:", energy_df.shape)
energy_df.head()

Shape after dropping nulls: (19724, 41)


,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,RH_5,T6,RH_6,T7,RH_7,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2,hour,day_of_week,month,day_of_month,is_weekend,Appliances_lag_1,Appliances_lag_3,Appliances_lag_6,Appliances_rolling_mean_6,Appliances_rolling_mean_12,T_out_RH_out_interaction,T1_RH_1_interaction,T2_RH_2_interaction
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-01-11 18:50:00,580,60,20.066667,46.396667,19.426667,44.400000,19.790000,44.826667,19.0,46.430000,17.10,55.000000,6.123333,87.993333,17.530000,44.263333,18.066667,48.633333,16.890000,45.290000,5.983333,734.433333,91.166667,5.833333,40.0,4.616667,8.827838,8.827838,18,0,1,11,0,230.0,60.0,50.0,176.666667,115.833333,545.480556,931.026444,862.544000
2016-01-11 19:00:00,430,50,20.133333,48.000000,19.566667,44.400000,19.890000,44.900000,19.0,46.363333,17.10,55.090000,6.123333,88.590000,17.823333,45.493333,18.066667,48.560000,16.963333,45.290000,6.000000,734.500000,91.000000,6.000000,40.0,4.600000,34.351142,34.351142,19,0,1,11,0,580.0,70.0,60.0,238.333333,146.666667,546.000000,966.400000,868.760000
2016-01-11 19:10:00,250,40,20.260000,52.726667,19.730000,45.100000,19.890000,45.493333,19.0,47.223333,17.10,55.163333,6.067500,88.215000,17.963333,46.160000,18.033333,48.666667,16.890000,45.326667,6.000000,734.616667,90.500000,6.000000,40.0,4.516667,19.205186,19.205186,19,0,1,11,0,430.0,230.0,60.0,270.000000,162.500000,543.000000,1068.242267,889.823000
2016-01-11 19:20:00,100,10,20.426667,55.893333,19.856667,45.833333,20.033333,47.526667,19.0,48.696667,17.10,55.500000,5.900000,88.156667,17.963333,45.533333,18.100000,49.193333,16.890000,45.345000,6.000000,734.733333,90.000000,6.000000,40.0,4.433333,38.492071,38.492071,19,0,1,11,0,250.0,580.0,60.0,276.666667,166.666667,540.000000,1141.714489,910.097222
2016-01-11 19:30:00,100,10,20.566667,53.893333,20.033333,46.756667,20.100000,48.466667,19.0,48.490000,17.15,56.042500,5.800000,88.366667,17.890000,44.926667,18.150000,49.200000,16.890000,45.326667,6.000000,734.850000,89.500000,6.000000,40.0,4.350000,24.884962,24.884962,19,0,1,11,0,100.0,430.0,70.0,281.666667,170.833333,537.000000,1108.406222,936.691889


#### 7) Saving feature-engineered dataset

In [14]:
save_dataframe(
    energy_df=energy_df,
    file_name="feature_engineered_energy_data.csv",
)